# Notebook 05: API Architecture and Deployment

This notebook describes how the four AI pipelines from the previous notebooks are integrated
into a production web application: a Flask backend serving a JavaScript frontend, packaged
in a Docker container and deployed on HuggingFace Spaces.

**What you will learn:**
- The overall system architecture: how the frontend, backend, and models interact
- The design of each Flask API endpoint and the data flow it manages
- How models are loaded and warmed up at startup to eliminate first-request latency
- The caching strategies used for the U-Net crop and Grad-CAM sub-models
- How the application is containerized with Docker for HuggingFace Spaces
- How model weights are kept out of the repository and downloaded at runtime

**Source code references:**
- `Backend/app.py` — Flask routes
- `Backend/model_loader.py` — model loading and warmup
- `Backend/config.py` — environment and path configuration
- `Dockerfile` — container definition
- `run.sh` — container entrypoint

## 1. System Architecture

```
Browser (Frontend)
  |
  |  HTTP POST (multipart/form-data: image + params)
  v
Flask App  (Backend/app.py)
  |
  |-- /predict              -> SEG_300 pipeline
  |-- /predict_512_fast     -> CNN + Hybrid pipeline
  |-- /predict_512_gradcam  -> Grad-CAM visualization
  |-- /predict_300_gradcam  -> Grad-CAM on 300px crop
  |-- /predict_yolo         -> YOLO + Calcium pipeline
  |-- /stats                -> Usage statistics
  |-- /  (static)           -> Frontend HTML/JS/CSS
  |
  v
AI Logic Layer
  |
  |-- image_utils.py        -> DICOM/image loading, normalization
  |-- ai_logic.py           -> pipeline_300, crop_with_unet, get_gradcam_image
  |-- Calcium_Pipeline.py   -> CalciumPipeline.run()
  |
  v
Loaded Models (in-process, shared across requests)
  |
  |-- cnn_model / combined_cnn  (TensorFlow/Keras, .h5)
  |-- ml_model                  (scikit-learn, .joblib)
  |-- unet_model                (PyTorch, .ckpt)
  |-- class_300_model           (TensorFlow/Keras, .h5)
  |-- yolo_re / yolo_ri         (Ultralytics YOLO, .pt)
  |-- calcium_unet              (PyTorch, .pth)
  |-- calcium_classifier        (scikit-learn, .joblib)
```

All models live in the same process as the Flask server. This avoids inter-process
communication overhead but means the application is CPU/memory bound. For a production
deployment with high traffic, a separate model server (e.g., TorchServe or TFServing)
behind a load balancer would be more appropriate.

## 2. Flask Endpoint Design

All prediction endpoints follow the same contract:
- **Method:** POST
- **Request:** `multipart/form-data` with a `file` field and optional parameters
- **Response:** JSON object with a `success` boolean and result fields
- **Error handling:** any exception returns `{"error": str(e)}` with HTTP 500

### Endpoint map

| Endpoint | Pipeline | Parameters | Key outputs |
|----------|----------|------------|-------------|
| `POST /predict` | SEG_300 | `file`, `threshold` | `image_original`, `image_segmented`, `image_cropped`, `result_seg` |
| `POST /predict_512_fast` | CNN + Hybrid | `file`, `threshold`, `age`, `sex` | `image_original`, `result_cnn`, `result_hybrid` |
| `POST /predict_512_gradcam` | Grad-CAM (512) | `file` | `gradcam_preview` |
| `POST /predict_300_gradcam` | Grad-CAM (300) | `file` | `gradcam_preview` |
| `POST /predict_yolo` | YOLO + Calcium | `file` | `image_yolo`, `image_calcium`, `prediction`, `model_info` |
| `GET /stats` | Statistics | — | usage counters per mode and laterality |
| `GET /` | Static | — | `index.html` |
| `GET /<path>` | Static | — | Any file in `../Frontend/` |

All image results are encoded as **base64 PNG strings** embedded directly in the JSON response.
This avoids the need for a separate static file server or temporary file management, at the
cost of larger JSON payloads.

## 3. Model Loading and Startup Warmup

All models are loaded once when `model_loader.py` is first imported (which happens at Flask
startup). Failed loads are caught and logged, and the corresponding model variable is set to
`None`. Every downstream function checks for `None` before calling the model, so a missing
model weight gracefully degrades the corresponding endpoint rather than crashing the server.

### Warmup passes

Deep learning frameworks (TensorFlow and PyTorch) perform several expensive operations on the
first forward pass through a model:
- TensorFlow compiles computation graphs (XLA compilation)
- PyTorch JIT-compiles kernels
- Both allocate GPU memory and load CUDA libraries

This makes the first real request visibly slow (several seconds). To avoid this user-facing
latency, `model_loader.py` runs a **warmup pass** with a zero-filled dummy array immediately
after each model is loaded:

```python
# Warmup for TensorFlow models
_dummy_512 = np.zeros((1, 512, 512, 3), dtype=np.float32)
if combined_cnn:
    combined_cnn.predict(_dummy_512, verbose=0)
if class_300_model:
    class_300_model.predict(np.zeros((1, 300, 300, 3), dtype=np.float32), verbose=0)

# Warmup for YOLO models
_dummy_896 = np.zeros((896, 896, 3), dtype=np.uint8)
if yolo_re:
    yolo_re.predict(_dummy_896, imgsz=896, verbose=False)
```

The warmup adds a few seconds to startup time but ensures all subsequent user requests
respond at their steady-state latency.

## 4. Caching Strategies

Two in-memory caches reduce redundant computation:

### 4.1 U-Net crop cache (TTL cache)

When a user uploads an image and clicks multiple analysis buttons in quick succession,
multiple POST requests arrive with the same image bytes. Each endpoint that uses the
U-Net crop would rerun the segmentation model.

The crop cache in `ai_logic.py` stores `(crop, mask, timestamp)` tuples keyed by a
fast MD5 fingerprint of the first 4096 bytes of the flattened image array plus its shape.
Entries expire after 180 seconds. A full MD5 of the image is avoided because hashing
a multi-megapixel medical image on every request would add measurable latency.

In [ ]:
import hashlib
import time
import numpy as np

# Minimal reproduction of the crop cache mechanism
_crop_cache = {}
CACHE_TTL = 180  # seconds

def fingerprint(img: np.ndarray) -> str:
    sample = img.ravel()[:4096].tobytes()
    return hashlib.md5(sample + str(img.shape).encode()).hexdigest()

def cached_crop(img, compute_fn):
    key = fingerprint(img)
    entry = _crop_cache.get(key)
    if entry and time.time() - entry["ts"] < CACHE_TTL:
        print("Cache HIT — returning stored crop")
        return entry["crop"]
    print("Cache MISS — running segmentation model")
    crop = compute_fn(img)
    _crop_cache[key] = {"crop": crop, "ts": time.time()}
    return crop


# Simulate two requests with the same image
dummy_image = np.random.randint(0, 256, (512, 512), dtype=np.uint8)
mock_compute = lambda img: img[100:400, 100:400]  # placeholder

result1 = cached_crop(dummy_image, mock_compute)
result2 = cached_crop(dummy_image, mock_compute)  # should be a cache hit

print(f"\nBoth results identical: {np.array_equal(result1, result2)}")

### 4.2 Grad-CAM model cache (dict keyed by model id)

Building the Grad-CAM sub-models (splitting the VGG19 at `block5_conv4`) involves graph
traversal and model construction. This is done once per model instance and cached in a
module-level dictionary using `id(model)` as the key:

```python
_gradcam_cache = {}  # {id(model): (conv_model, classifier_model)}

def _build_gradcam_models(model):
    key = id(model)  # stable for the lifetime of the process
    if key in _gradcam_cache:
        return _gradcam_cache[key]
    # ... build sub-models ...
    _gradcam_cache[key] = (conv_model, classifier_model)
    return conv_model, classifier_model
```

Since model objects are never recreated between requests, `id(model)` is a stable and
unique key for the lifetime of the server process.

## 5. Configuration and Secrets Management

The application must work in two environments:

1. **Local development:** Model files are downloaded manually and their paths are stored in
   a `.env.local` file that is listed in `.gitignore` and never committed.

2. **HuggingFace Spaces production:** Secrets (Google Drive IDs, filenames) are injected
   as environment variables through the Spaces "Variables and secrets" settings panel.

`Backend/config.py` handles both cases transparently:

```python
def _load_local_env():
    if os.environ.get("SPACE_ID"):   # HuggingFace sets this automatically
        return                        # use Spaces secrets, do nothing
    env_path = os.path.join(BASE_DIR, ".env.local")
    # ... read and export key=value pairs ...

_load_local_env()
```

Model weights are never stored in the repository. Instead, `model_loader.py` downloads
them from Google Drive using `gdown` at startup, skipping the download if the file
already exists and has non-zero size:

```python
def download_file(path, drive_id, name):
    if not path or not drive_id:
        return
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return  # already downloaded
    gdown.download(f"https://drive.google.com/uc?id={drive_id}", path)
```

This pattern keeps the Docker image small and the repository clean, while allowing
weights to be updated independently of the code.

## 6. Docker and HuggingFace Spaces

### Dockerfile walkthrough

```dockerfile
FROM python:3.10-slim          # Minimal Python base image

WORKDIR /app

# System libraries needed by OpenCV
RUN apt-get update && apt-get install -y \
    libglib2.0-0 libgl1 \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt  # install Python deps

COPY Backend/ ./Backend/        # Python source + config
COPY Frontend/ ./Frontend/      # HTML, CSS, JS
COPY run.sh .

EXPOSE 7860                     # HuggingFace Spaces uses port 7860

RUN chmod +x run.sh
CMD ["./run.sh"]                # Starts: python Backend/app.py
```

### Why python:3.10-slim?
The slim variant omits many development tools and documentation files from the full Python
image, resulting in a smaller final image. Libraries that need native compilation
(OpenCV, PyTorch) are installed via pip wheels which include their own native binaries.
Only `libglib2.0-0` and `libgl1` need to be installed at the OS level (OpenCV runtime deps).

### HuggingFace Spaces
HuggingFace Spaces is a free hosting platform for ML demos. When configured with the
Docker SDK, it:
1. Builds the Docker image from the repository's `Dockerfile`
2. Starts the container, mapping port 7860 to the public URL
3. Injects environment variables (secrets) defined in the Spaces settings
4. Provides `SPACE_ID` which `config.py` uses to detect the production environment

Model weights are downloaded to `/app/Backend/` at first startup and cached on the
container's filesystem for the duration of the Space session.

## 7. Testing the API Locally

Once the Flask server is running (`python Backend/app.py` or `./run.sh`), the endpoints
can be tested programmatically using the `requests` library.

In [ ]:
import requests
import base64
import numpy as np
import cv2
import matplotlib.pyplot as plt
import json

BASE_URL = "http://localhost:7860"
IMAGE_PATH = "../Frontend/Images/shoulder_x-ray.jpg"

def decode_b64_image(b64_string):
    """Decode a base64 PNG string returned by the API into a numpy array."""
    prefix = "data:image/png;base64,"
    if b64_string.startswith(prefix):
        b64_string = b64_string[len(prefix):]
    raw = base64.b64decode(b64_string)
    arr = np.frombuffer(raw, np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


# Test /predict (SEG_300 pipeline)
try:
    with open(IMAGE_PATH, "rb") as f:
        response = requests.post(
            f"{BASE_URL}/predict",
            files={"file": ("shoulder.jpg", f, "image/jpeg")},
            data={"threshold": 0.5},
            timeout=60
        )

    data = response.json()
    print("Status:", response.status_code)
    print("Success:", data.get("success"))
    print("Laterality:", data.get("laterality"))
    print("SEG result:", data.get("result_seg"))

    if data.get("image_original"):
        img_orig = decode_b64_image(data["image_original"])
        plt.figure(figsize=(5, 5))
        plt.imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
        plt.title(f"API response: {data.get('result_seg', {}).get('diagnosis', '')}")
        plt.axis("off")
        plt.show()

except requests.exceptions.ConnectionError:
    print("Server not running. Start the app first:")
    print("  cd c:/Users/sergi/TFG && python Backend/app.py")

In [ ]:
# Test /predict_512_fast (CNN + Hybrid)
try:
    with open(IMAGE_PATH, "rb") as f:
        response = requests.post(
            f"{BASE_URL}/predict_512_fast",
            files={"file": ("shoulder.jpg", f, "image/jpeg")},
            data={"threshold": 0.5, "age": 65, "sex": 0},
            timeout=60
        )

    data = response.json()
    print("CNN result  :", data.get("result_cnn"))
    print("Hybrid result:", data.get("result_hybrid"))

except requests.exceptions.ConnectionError:
    print("Server not running.")

In [ ]:
# Test /stats
try:
    response = requests.get(f"{BASE_URL}/stats", timeout=10)
    stats = response.json()
    print(json.dumps(stats, indent=2))
except requests.exceptions.ConnectionError:
    print("Server not running.")

## 8. Usage Statistics

The `/stats` endpoint returns aggregated usage counters stored in `Backend/stats.json`.
The `stats_manager.py` module updates the counters atomically after each successful prediction:

```python
update_stats(mode, laterality, positive)
# mode: "seg" or "cnn"
# laterality: "Right" or "Left"
# positive: True if the model predicts tendinopathy
```

The counters are persisted to disk after each update so they survive server restarts.
On HuggingFace Spaces, the container filesystem is ephemeral (resets on restart), so
stats are lost between deployments — this is acceptable for a research prototype.

For a production system, these counters would be stored in a persistent database
(PostgreSQL, SQLite with volume mount, or a managed cloud database).

## 9. Summary: Full System at a Glance

```
Repository structure
  Backend/          <- Python source (Flask, AI logic, model loaders)
  Frontend/         <- HTML, CSS, JavaScript (no build step required)
  Dockerfile        <- Container definition for HuggingFace Spaces
  requirements.txt  <- All Python dependencies
  run.sh            <- Entrypoint: python Backend/app.py

Deployment flow
  git push github main
    |
    v
  (manual) sync to HuggingFace Spaces repo
    |
    v
  Spaces builds Docker image from Dockerfile
    |
    v
  Container starts: model_loader.py downloads weights from Google Drive
    |
    v
  Flask serves at https://huggingface.co/spaces/egiiddo/TFG (port 7860)
```

### Design trade-offs

| Decision | Alternative | Reason chosen |
|----------|-------------|---------------|
| Single Flask process with all models | Separate model server | Simpler deployment on free-tier Spaces |
| Weights downloaded at runtime | Stored in repo with Git LFS | Avoids LFS bandwidth costs; weights update independently of code |
| Base64 images in JSON | Presigned URLs / static file server | No storage infrastructure needed; works entirely in-memory |
| In-memory TTL cache | Redis / Memcached | No external service dependency; sufficient for single-instance demo |
| python:3.10-slim | GPU-enabled CUDA image | No GPU available on free Spaces tier; keeps image size down |

These decisions optimize for **simplicity and zero infrastructure cost** at the expense of
scalability. For a clinical deployment, each trade-off would need to be revisited.